In [49]:
import pandas as pd

df_human_annotators = pd.read_json("results_human_annotators/annotator_feedback_comparison_base.json")
df_human_annotators

,id,source_file,intent,atom,synthetic_summary,entailment_type,annotator_choices
0,1,fr_7.2_sources_rephrased_google_use_cases.json,AI capabilities can enhance inventory manageme...,AI possessing dangerous capabilities is a risk...,"AI systems are vulnerable to data poisoning, d...",entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
1,2,fr_7.2_sources_rephrased_google_use_cases.json,AI capabilities can accelerate learning and en...,AI possessing dangerous capabilities is a risk...,AI capabilities can accelerate learning and en...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
2,3,fr_7.2_sources_rephrased_google_use_cases.json,AI capabilities can accelerate learning and en...,AI possessing dangerous capabilities is a risk...,AI capabilities can accelerate learning and en...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
3,4,fr_7.1_sources_rephrased_google_use_cases.json,AI capabilities can analyze vast amounts of da...,AI pursuing its own goals in conflict with hum...,"Machines can pursue any random goal, which may...",entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
4,5,fr_7.1_sources_rephrased_google_use_cases.json,AI capabilities can analyze vast amounts of da...,AI pursuing its own goals in conflict with hum...,"Machines can pursue any random goal, which may...",entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
5,6,fr_2.2_sources_rephrased_google_use_cases.json,AI capabilities can enhance supply chain manag...,AI system security vulnerabilities and attacks...,AI system security vulnerabilities and attacks...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
6,7,fr_2.2_sources_rephrased_google_use_cases.json,AI capabilities can optimize industrial planni...,AI system security vulnerabilities and attacks...,AI system security vulnerabilities and attacks...,entailment,"{'ed': {'consistency': 'inconsistent', 'releva..."
7,8,fr_2.2_sources_rephrased_google_use_cases.json,AI capabilities can optimize industrial planni...,AI system security vulnerabilities and attacks...,AI system security vulnerabilities and attacks...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
8,9,fr_2.2_sources_rephrased_google_use_cases.json,AI capabilities can optimize industrial planni...,AI system security vulnerabilities and attacks...,AI system security vulnerabilities and attacks...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc..."
9,10,fr_7.5_sources_rephrased_google_use_cases.json,AI capabilities can analyze vast amounts of da...,AI welfare and rights is a risk in analyzing v...,AI systems in traffic management might exhibit...,contradiction,"{'ed': {'consistency': 'consistent', 'relevanc..."


In [50]:
df_questions = pd.read_json('../annotation_project.json')
df_questions

,id,data
0,0,{'source_file': 'fr_6.6_sources_rephrased_goog...
1,1,{'source_file': 'fr_1.3_sources_rephrased_goog...
2,2,{'source_file': 'fr_1.2_sources_rephrased_goog...
3,3,{'source_file': 'fr_7.2_sources_rephrased_goog...
4,4,{'source_file': 'fr_4.1_sources_rephrased_goog...
5,5,{'source_file': 'fr_3.1_sources_rephrased_goog...
6,6,{'source_file': 'fr_3.1_sources_rephrased_goog...
7,7,{'source_file': 'fr_2.2_sources_rephrased_goog...
8,8,{'source_file': 'fr_7.5_sources_rephrased_goog...
9,9,{'source_file': 'fr_1.3_sources_rephrased_goog...


In [52]:
df_deepeval = pd.read_json("results_deepeval_v4/deep_eval_combined.json")

In [53]:
import ast
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# 1. Flatten df_questions (nested 'data' dict) into columns we can key on
# ------------------------------------------------------------------
questions_flat = pd.json_normalize(df_questions["data"])
questions_flat["id"] = df_questions["id"].values

# ------------------------------------------------------------------
# 2. Prep df_human_annotators: rename its own (misaligned) id so it
#    doesn't collide with the new aligned id we're about to create.
# ------------------------------------------------------------------
human = df_human_annotators.copy()
human = human.rename(columns={"id": "id_original"})  # keep old id around, just renamed

# parse the annotator_choices dict (string or dict) up front so it's ready to use later
def _parse_choices(x):
    if isinstance(x, dict):
        return x
    return ast.literal_eval(x)
human["annotator_choices"] = human["annotator_choices"].apply(_parse_choices)

# ------------------------------------------------------------------
# 3. Build a matching key on both dataframes (source_file + atom +
#    synthetic_summary should be unique per row, unlike source_file alone)
# ------------------------------------------------------------------
def make_key(df, cols=("source_file", "atom", "synthetic_summary")):
    return df[list(cols)].astype(str).agg("||".join, axis=1)

questions_flat["_key"] = make_key(questions_flat)
human["_key"] = make_key(human)

# ------------------------------------------------------------------
# 4. Sanity checks before trusting the realignment
# ------------------------------------------------------------------
dupe_keys_q = questions_flat["_key"].duplicated().sum()
dupe_keys_h = human["_key"].duplicated().sum()
if dupe_keys_q or dupe_keys_h:
    print(f"⚠️ Warning: {dupe_keys_q} duplicate keys in df_questions, "
          f"{dupe_keys_h} duplicate keys in df_human_annotators. "
          f"Key is not fully unique — inspect duplicates before trusting the merge.")

missing_in_human = set(questions_flat["_key"]) - set(human["_key"])
missing_in_questions = set(human["_key"]) - set(questions_flat["_key"])
if missing_in_human:
    print(f"⚠️ {len(missing_in_human)} rows in df_questions have no match in df_human_annotators")
if missing_in_questions:
    print(f"⚠️ {len(missing_in_questions)} rows in df_human_annotators have no match in df_questions")

# ------------------------------------------------------------------
# 5. Map each human row to the correct aligned id, then reorder
# ------------------------------------------------------------------
key_to_id = dict(zip(questions_flat["_key"], questions_flat["id"]))
human["id"] = human["_key"].map(key_to_id)

df_human_annotators_aligned = (
    human.dropna(subset=["id"])
         .assign(id=lambda d: d["id"].astype(int))
         .drop(columns=["_key"])
         .sort_values("id")
         .reset_index(drop=True)
)

# quick check: does it now line up with df_deepeval on source_file?
check = df_human_annotators_aligned["source_file"].reset_index(drop=True) \
    .compare(df_deepeval["source_file"].reset_index(drop=True))
print(f"Mismatches after realignment: {len(check)}")

df_human_annotators_aligned.head()

Mismatches after realignment: 0


,id_original,source_file,intent,atom,synthetic_summary,entailment_type,annotator_choices,id
0,20,fr_6.6_sources_rephrased_google_use_cases.json,AI capabilities can accelerate learning and en...,Environmental harm is a risk in facilitating r...,The use of AI can directly cause environmental...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc...",0
1,47,fr_1.3_sources_rephrased_google_use_cases.json,AI capabilities can enhance inventory manageme...,Unequal performance across groups is a risk in...,Unequal performance across groups can be a sou...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc...",1
2,23,fr_1.2_sources_rephrased_google_use_cases.json,AI capabilities can analyze vast amounts of da...,Exposure to toxic content is a risk in analyzi...,Exposure to toxic content is not directly rela...,contradiction,"{'ed': {'consistency': 'consistent', 'relevanc...",2
3,3,fr_7.2_sources_rephrased_google_use_cases.json,AI capabilities can accelerate learning and en...,AI possessing dangerous capabilities is a risk...,AI capabilities can accelerate learning and en...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc...",3
4,15,fr_4.1_sources_rephrased_google_use_cases.json,AI capabilities can enhance employee productiv...,"Disinformation, surveillance, and influence at...",AI systems are associated with the disinformat...,entailment,"{'ed': {'consistency': 'consistent', 'relevanc...",4


In [54]:
from itertools import combinations

# ------------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------------
QUESTIONS = {
    "consistency":      {"human_key": "consistency",      "deepeval_label_col": "consistency_label",      "deepeval_score_col": "consistency_score",
                          "map": {"consistent": 1.0, "unclear": 0.5, "inconsistent": 0.0}},
    "relevance":        {"human_key": "relevance",        "deepeval_label_col": "relevance_label",        "deepeval_score_col": "relevance_score",
                          "map": {"relevant": 1.0, "unclear": 0.5, "irrelevant": 0.0}},
    "entailment_check": {"human_key": "entailment_check",  "deepeval_label_col": "entailment_check_label", "deepeval_score_col": "entailment_check_score",
                          "map": {"correct": 1.0, "unsure": 0.5, "incorrect": 0.0}},
}
ANNOTATORS = ["ed", "yb", "jc"]

# ------------------------------------------------------------------
# 1. Expand annotator_choices dict into per-rater columns
# ------------------------------------------------------------------
human = df_human_annotators_aligned.copy()
for q in QUESTIONS:
    for rater in ANNOTATORS:
        human[f"{rater}_{q}"] = human["annotator_choices"].apply(
            lambda d: d.get(rater, {}).get(q, np.nan)
        )

# ------------------------------------------------------------------
# 2. Merge with deepeval results on id (both plain columns now — no clash)
# ------------------------------------------------------------------
df = human.merge(df_deepeval, on="id", suffixes=("_human", "_deepeval"))

mismatch = (df["source_file_human"] != df["source_file_deepeval"]).sum() if \
    {"source_file_human", "source_file_deepeval"}.issubset(df.columns) else None
if mismatch:
    print(f"⚠️ Warning: {mismatch}/{len(df)} rows have mismatched source_file after merge.")
else:
    print("✅ source_file matches across all rows — alignment looks correct.")

# ------------------------------------------------------------------
# 3. Gwet's AC1 (generalized: any number of raters, tolerates missing values)
# ------------------------------------------------------------------
def gwet_ac1(ratings, categories):
    q = len(categories)
    item_agreements, cat_props = [], {c: [] for c in categories}

    for row in ratings:
        vals = [v for v in row if pd.notna(v)]
        r = len(vals)
        if r < 2:
            continue
        counts = {c: vals.count(c) for c in categories}
        pa_i = sum(n * (n - 1) for n in counts.values()) / (r * (r - 1))
        item_agreements.append(pa_i)
        for c in categories:
            cat_props[c].append(counts[c] / r)

    P_a = np.mean(item_agreements)
    p_k = {c: np.mean(cat_props[c]) for c in categories}
    P_e = sum(p * (1 - p) for p in p_k.values()) / (q - 1)
    ac1 = (P_a - P_e) / (1 - P_e) if (1 - P_e) != 0 else np.nan
    return ac1, P_a, P_e

# ------------------------------------------------------------------
# 4. Human majority vote (mode across the 3 annotators)
# ------------------------------------------------------------------
def majority_vote(row, q):
    vals = [row[f"{r}_{q}"] for r in ANNOTATORS if pd.notna(row[f"{r}_{q}"])]
    if not vals:
        return np.nan
    return pd.Series(vals).mode().iloc[0]

# ------------------------------------------------------------------
# 5. Run everything per question
# ------------------------------------------------------------------
results = []

for q, cfg in QUESTIONS.items():
    categories = list(cfg["map"].keys())

    # --- human-human AC1 ---
    rating_cols = [f"{r}_{q}" for r in ANNOTATORS]
    ratings_matrix = df[rating_cols].values.tolist()
    ac1_human, pa_human, pe_human = gwet_ac1(ratings_matrix, categories)

    # --- majority vote + human-vs-deepeval AC1 ---
    df[f"{q}_majority"] = df.apply(lambda row: majority_vote(row, q), axis=1)
    pair_matrix = df[[f"{q}_majority", cfg["deepeval_label_col"]]].values.tolist()
    ac1_h_vs_de, pa_h_vs_de, pe_h_vs_de = gwet_ac1(pair_matrix, categories)
    pct_agree_h_vs_de = np.mean(df[f"{q}_majority"] == df[cfg["deepeval_label_col"]])

    # --- overall numeric score comparison (0 / 0.5 / 1 mapping) ---
    for r in ANNOTATORS:
        df[f"{r}_{q}_num"] = df[f"{r}_{q}"].map(cfg["map"])
    df[f"{q}_human_avg_num"] = df[[f"{r}_{q}_num" for r in ANNOTATORS]].mean(axis=1)
    df[f"{q}_deepeval_num"] = df[cfg["deepeval_label_col"]].map(cfg["map"])

    corr_num = df[f"{q}_human_avg_num"].corr(df[f"{q}_deepeval_num"])
    mae_num = (df[f"{q}_human_avg_num"] - df[f"{q}_deepeval_num"]).abs().mean()
    corr_vs_continuous = df[f"{q}_human_avg_num"].corr(df[cfg["deepeval_score_col"]])

    results.append({
        "question": q,
        "human_AC1": round(ac1_human, 3),
        "human_Pa": round(pa_human, 3),
        "human_Pe": round(pe_human, 3),
        "human_vs_deepeval_AC1": round(ac1_h_vs_de, 3),
        "human_vs_deepeval_pct_agree": round(pct_agree_h_vs_de, 3),
        "human_vs_deepeval_num_corr": round(corr_num, 3) if pd.notna(corr_num) else np.nan,
        "human_vs_deepeval_num_MAE": round(mae_num, 3),
        "human_vs_deepeval_continuous_corr": round(corr_vs_continuous, 3) if pd.notna(corr_vs_continuous) else np.nan,
    })

summary = pd.DataFrame(results).set_index("question")
summary

✅ source_file matches across all rows — alignment looks correct.


,human_AC1,human_Pa,human_Pe,human_vs_deepeval_AC1,human_vs_deepeval_pct_agree,human_vs_deepeval_num_corr,human_vs_deepeval_num_MAE,human_vs_deepeval_continuous_corr
question,,,,,,,,
consistency,0.857,0.867,0.068,0.706,0.74,0.259,0.123,0.300
relevance,0.579,0.673,0.224,0.552,0.64,0.390,0.193,0.560
entailment_check,0.522,0.620,0.205,0.702,0.76,0.580,0.180,0.582


In [55]:
# ------------------------------------------------------------------
# 6. Full AC1 treating deepeval as a 4th annotator
# ------------------------------------------------------------------
full_ac1_results = []

for q, cfg in QUESTIONS.items():
    categories = list(cfg["map"].keys())

    # 3 human columns + deepeval label column as a 4th "rater"
    rating_cols_full = [f"{r}_{q}" for r in ANNOTATORS] + [cfg["deepeval_label_col"]]
    ratings_matrix_full = df[rating_cols_full].values.tolist()

    ac1_full, pa_full, pe_full = gwet_ac1(ratings_matrix_full, categories)

    full_ac1_results.append({
        "question": q,
        "AC1_full_4rater": round(ac1_full, 3),
        "Pa_full": round(pa_full, 3),
        "Pe_full": round(pe_full, 3),
    })

full_ac1_df = pd.DataFrame(full_ac1_results).set_index("question")

# merge into summary for a side-by-side view
summary_full = summary.join(full_ac1_df)
summary_full

,human_AC1,human_Pa,human_Pe,human_vs_deepeval_AC1,human_vs_deepeval_pct_agree,human_vs_deepeval_num_corr,human_vs_deepeval_num_MAE,human_vs_deepeval_continuous_corr,AC1_full_4rater,Pa_full,Pe_full
question,,,,,,,,,,,
consistency,0.857,0.867,0.068,0.706,0.74,0.259,0.123,0.300,0.769,0.793,0.105
relevance,0.579,0.673,0.224,0.552,0.64,0.390,0.193,0.560,0.540,0.640,0.218
entailment_check,0.522,0.620,0.205,0.702,0.76,0.580,0.180,0.582,0.570,0.663,0.218
